# Reactivity Pilot — Statistical Inference (Stage A / Stage B)

범위: `reactivity_pilot.ipynb`가 만든 `reactivity_logits_3tokens.csv`를 입력으로, position 효과를
통제한 뒤에도 useState-form 효과가 0보다 유의하게 큰지 검정하고(Stage A), position·useState-form·
declaration-order·identity bias 네 효과의 크기를 회귀로 동시에 분리한다(Stage B). activation
patching, 추가 logit 측정은 이 노트북에 포함하지 않는다 — 이미 측정된 CSV만 쓴다.

## 실행 환경

GPU·모델 가중치·TransformerLens가 필요 없다. `pandas`/`numpy`/`scipy`/`statsmodels`만 쓴다.
이 노트북의 핵심 계산 로직(pair-mean 검정, D 복원, predictor 도출, mixed model/clustered OLS)은
실제 `reactivity_logits_3tokens.csv` 데이터로 별도 환경에서 사전 검증되었다 — predictor들이
설계상 완전히 직교(상호 상관 ≈ 0)함을 확인했고, mixed model은 두 모델 모두 수렴했다. 다만 이
노트북 파일 자체는 실행되지 않은 채로 전달한다 — 출력 파일은 연구자가 셀을 위에서 아래로 돌려
직접 만든다.

## 용어·주장 톤에 대한 중요한 주의 (이전 노트북과 동일)

`reactive_var`/`stable_var`, 그리고 이 노트북에서 새로 쓰는 `dep`/`alt` identity 기준 표현 모두
useState/const **선언 형태**에 붙인 라벨이며, reactivity 그 자체를 측정한 것이 아니다. 다음과
같은 표현은 코드·markdown·출력 어디에도 쓰지 않는다: "LLM이 React reactivity를 이해한다",
"React 의미론 회로". 검정에서 유의한 양의 효과가 나와도 허용 claim은 **"useState-form /
declared-reactivity sensitivity"**까지이며, setter(`setX`)·destructuring 구조가 useState 선언에
항상 동반되므로 reactivity 자체와 분리되지 않는다는 한계가 남는다(§Limitation에서 다시 명시).

## 분석 두 단계

- **Stage A (pair-level 검정):** pair당 8 cell은 `reactive_body_pos`가 first 4 / second 4로
  균형 잡혀 있다. 사실은 이보다 더 강하다 — `state_role`(어느 식별자가 useState인지)도, `decl_order`
  도, `body_order`도 pair당 4/4로 균형 잡힌 완전요인설계다. 따라서 pair 안에서 8 cell의
  `LD_reactive`를 평균하면 position 효과뿐 아니라 declaration-order 효과와 "이 pair의 두 식별자
  중 어느 쪽이 우연히 더 흔한 단어인가" 같은 identity-specific bias까지 상쇄되고,
  useState-form(역할) 효과만 남는다. Stage A는 이 pair-mean이 0보다 큰지를 검정한다.
- **Stage B (회귀 분해):** Stage A의 pair-mean 검정은 네 가지 효과를 뭉쳐서 "평균 후 잔차"로만
  보여준다. Stage B는 같은 데이터를 dep/alt identity 기준 종속변수 `D`로 재구성하고, position·
  useState-form·decl_order 세 predictor로 회귀해 각 효과의 **크기**를 따로 추정한다. pair 내
  8 cell은 독립이 아니므로 pair를 random effect로 둔 mixed model을 main으로, pair-clustered SE를
  쓴 OLS를 비교용으로 함께 보고한다.


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import bootstrap, rankdata, t as tdist, ttest_1samp, wilcoxon

try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
except ImportError as e:
    raise ImportError(
        "statsmodels가 필요하다 (`pip install statsmodels`). Stage B의 MixedLM/clustered OLS에 쓴다."
    ) from e

RNG_SEED = 0  # bootstrap CI 재현성을 위해 고정


## Configuration

`reactivity_pilot.ipynb`가 같은 디렉터리에 만든 CSV를 입력으로 쓴다.

**확인할 것:** 행 수가 720(45 pair × 8 cell × 2 모델)인지, 두 모델 키가 정확히 일치하는지,
필요한 컬럼이 모두 있는지.


In [2]:
INPUT_CSV = Path("reactivity_logits_3tokens.csv")

MODEL_KEYS = ["llama3.2-1b", "gemma3-1b-pt"]
MODEL_LABELS = {"llama3.2-1b": "Llama 3.2 1B", "gemma3-1b-pt": "Gemma 3 1B-pt"}

REQUIRED_COLS = [
    "model", "pair_id", "cell_id", "state_role", "decl_order", "body_order",
    "reactive_var", "stable_var", "reactive_body_pos",
    "logit_reactive", "logit_stable", "LD_reactive",
]

df = pd.read_csv(INPUT_CSV)

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing_cols, f"missing required columns: {missing_cols}"

assert set(df["model"].unique()) == set(MODEL_KEYS), (
    f"unexpected model keys: {sorted(df['model'].unique())} vs expected {sorted(MODEL_KEYS)}"
)

n_pairs = df["pair_id"].nunique()
assert len(df) == len(MODEL_KEYS) * n_pairs * 8, (
    f"expected {len(MODEL_KEYS)} x {n_pairs} x 8 = {len(MODEL_KEYS) * n_pairs * 8} rows, got {len(df)}"
)

print(f"loaded {INPUT_CSV}: {df.shape[0]} rows, {n_pairs} pairs, models={sorted(df['model'].unique())}")


loaded reactivity_logits_3tokens.csv: 720 rows, 45 pairs, models=['gemma3-1b-pt', 'llama3.2-1b']


## Step 1 — Stage 0: 균형설계 검증

Stage A의 "평균하면 position이 상쇄된다"는 논리와 Stage B의 predictor 직교성 모두, pair당 8 cell이
`state_role`/`decl_order`/`body_order` 세 binary factor에 대해 정확히 4/4로 균형 잡힌 완전요인
설계라는 전제에 의존한다. 이 전제를 모델별·pair별로 직접 확인한다.

**확인할 것:** 아래 셀이 assert 없이 통과하면 모든 pair가 실제로 2×2×2 완전요인이라는 뜻이다.
실패하면 이후 Stage A/B의 "상쇄" 논리와 회귀 해석이 깨지므로, 이 시점에서 멈추고 데이터 자체를
다시 봐야 한다.


In [3]:
def check_balance(sub, model_key):
    bad = []
    for pair_id, g in sub.groupby("pair_id"):
        if len(g) != 8:
            bad.append((pair_id, "n_cells", len(g)))
            continue
        for col in ["state_role", "decl_order", "body_order", "reactive_body_pos"]:
            counts = g[col].value_counts()
            if len(counts) != 2 or sorted(counts.tolist()) != [4, 4]:
                bad.append((pair_id, col, counts.to_dict()))
    return bad


for model_key in MODEL_KEYS:
    sub = df[df["model"] == model_key]
    bad = check_balance(sub, model_key)
    assert not bad, f"{model_key}: unbalanced pairs found: {bad[:10]}"
    print(f"{model_key}: {sub['pair_id'].nunique()} pairs, all balanced 4/4 on "
          f"state_role/decl_order/body_order/reactive_body_pos")


llama3.2-1b: 45 pairs, all balanced 4/4 on state_role/decl_order/body_order/reactive_body_pos
gemma3-1b-pt: 45 pairs, all balanced 4/4 on state_role/decl_order/body_order/reactive_body_pos


## Step 2 — Stage A 준비: pair-mean 계산

각 (model, pair)에 대해 8 cell의 `LD_reactive` 평균을 구한다. Step 1에서 확인했듯 이 8 cell은
`state_role`/`decl_order`/`body_order` 모두 4/4로 균형 잡혀 있으므로, 이 평균은 position·
decl_order·identity bias가 상쇄된 뒤의 useState-form(역할) 효과를 추정한다.

**확인할 것:** 모델당 정확히 45개 pair-mean 값이 나오는지.


In [4]:
pair_means = {}
for model_key in MODEL_KEYS:
    sub = df[df["model"] == model_key]
    pm = sub.groupby("pair_id")["LD_reactive"].mean()
    assert len(pm) == n_pairs, f"{model_key}: expected {n_pairs} pair-means, got {len(pm)}"
    pair_means[model_key] = pm
    print(f"{model_key}: {len(pm)} pair-means, range=({pm.min():.3f}, {pm.max():.3f})")


llama3.2-1b: 45 pair-means, range=(0.583, 1.724)
gemma3-1b-pt: 45 pair-means, range=(-0.351, 1.176)


## Step 3 — Stage A 검정: pair-mean LD_reactive > 0

가설은 방향성이 있다(useState-form 쪽이 더 높다는 양의 방향) — **단측(alternative="greater")이
기본**이며, 참고용으로 양측 p값도 같이 낸다.

- **one-sample t-test** (모수, 정규성 가정): `scipy.stats.ttest_1samp`.
- **Wilcoxon signed-rank test** (비모수, 분포 가정에 덜 의존): `scipy.stats.wilcoxon`. pair-mean을
  "0 against"의 차이값으로 바로 취급한다(0인 값이 있으면 `zero_method="wilcox"` 기본값으로
  제외된다 — 부동소수점 평균이 정확히 0일 가능성은 거의 없지만 방어적으로 그대로 둔다).

효과크기:

- t-test → **Cohen's d** = mean / sd.
- Wilcoxon → **matched-pairs rank-biserial correlation** r = (W+ - W-) / (W+ + W-), W+/W-는 0이
  아닌 pair-mean들의 절댓값 순위 합을 양/음 부호로 나눈 것.

신뢰구간: mean의 **t 기반 95% CI**, median의 **bootstrap(BCa) 95% CI**(`n_resamples=10000`,
`random_state` 고정 — 재현 가능).


In [5]:
def rank_biserial_r(values):
    """Matched-pairs rank-biserial correlation for a one-sample/paired Wilcoxon test."""
    nz = values[values != 0]
    if len(nz) == 0:
        return float("nan")
    ranks = rankdata(np.abs(nz))
    w_pos = ranks[nz > 0].sum()
    w_neg = ranks[nz < 0].sum()
    return (w_pos - w_neg) / (w_pos + w_neg)


def stage_a_for_model(model_key, pm):
    values = pm.values
    n = len(values)
    mean_v, median_v = values.mean(), np.median(values)
    sd = values.std(ddof=1)
    sem = sd / np.sqrt(n)

    t_greater = ttest_1samp(values, popmean=0, alternative="greater")
    t_two = ttest_1samp(values, popmean=0, alternative="two-sided")
    cohend = mean_v / sd

    tcrit = tdist.ppf(0.975, df=n - 1)
    mean_ci_low, mean_ci_high = mean_v - tcrit * sem, mean_v + tcrit * sem

    w_greater = wilcoxon(values, alternative="greater", zero_method="wilcox")
    w_two = wilcoxon(values, alternative="two-sided", zero_method="wilcox")
    r_rb = rank_biserial_r(values)

    rng = np.random.default_rng(RNG_SEED)
    boot = bootstrap(
        (values,), np.median, confidence_level=0.95, n_resamples=10000,
        method="BCa", random_state=rng,
    )

    return {
        "model": model_key,
        "n_pairs": n,
        "pair_mean_mean": mean_v,
        "pair_mean_median": median_v,
        "pair_mean_sd": sd,
        "frac_pair_mean_pos": (values > 0).mean(),
        "t_stat": t_greater.statistic,
        "t_p_onesided_greater": t_greater.pvalue,
        "t_p_twosided": t_two.pvalue,
        "cohend": cohend,
        "mean_ci95_low": mean_ci_low,
        "mean_ci95_high": mean_ci_high,
        "wilcoxon_stat": w_greater.statistic,
        "wilcoxon_p_onesided_greater": w_greater.pvalue,
        "wilcoxon_p_twosided": w_two.pvalue,
        "rank_biserial_r": r_rb,
        "median_boot_ci95_low": boot.confidence_interval.low,
        "median_boot_ci95_high": boot.confidence_interval.high,
    }


stage_a_rows = [stage_a_for_model(mk, pair_means[mk]) for mk in MODEL_KEYS]
stage_a_df = pd.DataFrame(stage_a_rows)
stage_a_df.to_csv("reactivity_pairlevel_test.csv", index=False)
stage_a_df


,model,n_pairs,pair_mean_mean,pair_mean_median,pair_mean_sd,frac_pair_mean_pos,t_stat,t_p_onesided_greater,t_p_twosided,cohend,mean_ci95_low,mean_ci95_high,wilcoxon_stat,wilcoxon_p_onesided_greater,wilcoxon_p_twosided,rank_biserial_r,median_boot_ci95_low,median_boot_ci95_high
0,llama3.2-1b,45,1.071337,1.029885,0.243830,1.000000,29.474406,6.535587e-31,1.307117e-30,4.393785,0.998082,1.144591,1035.0,2.842171e-14,5.684342e-14,1.00000,0.957398,1.144455
1,gemma3-1b-pt,45,0.360482,0.376843,0.305302,0.844444,7.920632,2.622993e-10,5.245986e-10,1.180738,0.268759,0.452205,976.0,2.578645e-09,5.157290e-09,0.88599,0.277400,0.490579


## Step 4 — Stage A: 두 검정·두 모델 일치 여부

다중비교 인지: 여기서 핵심 가설검정은 "모델 2개 × 검정 2개(t-test, Wilcoxon)"의 4개 단측 p값이다.
보정은 하지 않고 raw p값을 그대로 보여준다 — 필요하면 연구자가 보정 방식(예: Bonferroni
α/4)을 정한다. 참고용으로 Bonferroni 기준 α 임계값만 같이 표시한다.

**확인할 것:** t-test와 Wilcoxon의 단측 p값 방향·크기가 두 모델에서 비슷한 패턴을 보이는지,
혹은 한쪽 모델/한쪽 검정에서만 유의한지.


In [6]:
N_COMPARISONS = len(MODEL_KEYS) * 2  # 2 models x {t-test, Wilcoxon}, one-sided
bonferroni_alpha = 0.05 / N_COMPARISONS

print(f"Bonferroni 참고 임계값 (alpha=0.05, m={N_COMPARISONS} comparisons): {bonferroni_alpha:.5f}")
print()

agreement_rows = []
for _, r in stage_a_df.iterrows():
    agreement_rows.append({
        "model": r["model"],
        "t_p_onesided": r["t_p_onesided_greater"],
        "wilcoxon_p_onesided": r["wilcoxon_p_onesided_greater"],
        "both_sig_at_0.05": bool(r["t_p_onesided_greater"] < 0.05 and r["wilcoxon_p_onesided_greater"] < 0.05),
        "both_sig_at_bonferroni": bool(
            r["t_p_onesided_greater"] < bonferroni_alpha and r["wilcoxon_p_onesided_greater"] < bonferroni_alpha
        ),
        "cohend": r["cohend"],
        "rank_biserial_r": r["rank_biserial_r"],
    })
agreement_df = pd.DataFrame(agreement_rows)
agreement_df


Bonferroni 참고 임계값 (alpha=0.05, m=4 comparisons): 0.01250



,model,t_p_onesided,wilcoxon_p_onesided,both_sig_at_0.05,both_sig_at_bonferroni,cohend,rank_biserial_r
0,llama3.2-1b,6.535587e-31,2.842171e-14,True,True,4.393785,1.00000
1,gemma3-1b-pt,2.622993e-10,2.578645e-09,True,True,1.180738,0.88599


## Step 5 — Stage B 준비: identity 기준 종속변수 `D`와 predictor 도출

Stage A의 `LD_reactive = logit(reactive_var) - logit(stable_var)`는 **role-relative**다(어느
물리적 식별자가 useState인지와 무관하게 항상 "reactive 쪽 - stable 쪽"). Stage B는 대신
**identity-relative** 종속변수를 쓴다.

```
D = logit(dep) - logit(alt)
```

CSV에는 `vars.dep`/`vars.alt` 문자열이 없지만 필요 없다 — `state_role`이 어느 쪽이 `dep`인지
이미 말해준다:

```
state_role == "dep_reactive"  →  reactive_var는 dep다  →  D = LD_reactive
state_role == "alt_reactive"  →  reactive_var는 alt다  →  D = -LD_reactive
```

predictor 세 개도 모두 **identity-relative(±1 코딩)**로 만든다. 이 코딩이 중요한 이유는 균형
2×2×2 설계에서 ±1 코딩 회귀계수가 정확히 "factorial main effect"(고-저 그룹 평균 차이의 절반)와
같아지기 때문이다 — 0/1 코딩이면 계수가 2배로 나와 비교가 어긋난다.

- **`pos_dep_first`**: `body_order`는 이미 dep/alt identity로 라벨링되어 있다(`"dep_first"`/
  `"alt_first"`) — role 보정 없이 그대로 ±1 변환하면 된다.
- **`role_dep_reactive`**: `state_role`에서 직접 — 이것이 바로 useState-form(역할) 효과를 잡는
  predictor다.
- **`decl_dep_first`**: `decl_order`는 `"reactive_first"`/`"stable_first"`로 **role-relative**하게
  라벨링되어 있어, `body_order`와 달리 그대로 못 쓴다. "`dep`가 먼저 선언되었는가"는
  `decl_order=="reactive_first"`인지와 `state_role=="dep_reactive"`인지가 **같은 경우에만** 참이다
  (XNOR). 둘 다 참이면 reactive=dep가 먼저 선언된 것이고, 둘 다 거짓이면 stable=dep가 먼저
  선언된 것이다.

아래 셀은 이 도출을 실제로 수행하고, 두 가지 독립적인 방식으로 같은 값을 다시 계산해 일치하는지
assert로 확인한다(코딩 실수가 있으면 여기서 바로 드러난다).


In [7]:
def build_stage_b_frame(sub):
    sub = sub.copy()
    dep_is_reactive = (sub["state_role"] == "dep_reactive").values

    # --- D: identity-relative dependent variable ---
    D = np.where(dep_is_reactive, sub["LD_reactive"].values, -sub["LD_reactive"].values)

    # cross-check: reconstruct logit_dep/logit_alt explicitly and recompute D the long way
    logit_dep = np.where(dep_is_reactive, sub["logit_reactive"].values, sub["logit_stable"].values)
    logit_alt = np.where(dep_is_reactive, sub["logit_stable"].values, sub["logit_reactive"].values)
    D_check = logit_dep - logit_alt
    assert np.allclose(D, D_check), "D reconstruction mismatch between role-relative and identity-relative paths"

    # --- pos_dep_first: body_order is already identity-anchored, no role correction needed ---
    pos_dep_first = np.where(sub["body_order"].values == "dep_first", 1, -1)

    # cross-check via reactive_body_pos (a genuinely separate column):
    # reactive_body_pos=="first"  <=>  XNOR(pos_dep_first==1, dep_is_reactive)
    reactive_first = (sub["reactive_body_pos"].values == "first")
    pos_dep_first_check = np.where(reactive_first == dep_is_reactive, 1, -1)
    assert np.array_equal(pos_dep_first, pos_dep_first_check), (
        "pos_dep_first mismatch between body_order-direct and reactive_body_pos-derived paths"
    )

    # --- role_dep_reactive: this is the useState-form / declared-reactivity predictor ---
    role_dep_reactive = np.where(dep_is_reactive, 1, -1)

    # --- decl_dep_first: decl_order is role-relative, needs XNOR with state_role ---
    decl_reactive_first = (sub["decl_order"].values == "reactive_first")
    decl_dep_first = np.where(decl_reactive_first == dep_is_reactive, 1, -1)

    # cross-check via the algebraic identity XNOR(a,b) == a*b in +-1 coding
    decl_sign = np.where(decl_reactive_first, 1, -1)
    decl_dep_first_check = decl_sign * role_dep_reactive
    assert np.array_equal(decl_dep_first, decl_dep_first_check), (
        "decl_dep_first mismatch between boolean-XNOR and signed-product paths"
    )

    sub["D"] = D
    sub["pos_dep_first"] = pos_dep_first
    sub["role_dep_reactive"] = role_dep_reactive
    sub["decl_dep_first"] = decl_dep_first
    return sub


stage_b_frames = {}
for model_key in MODEL_KEYS:
    sub = df[df["model"] == model_key]
    sb = build_stage_b_frame(sub)

    # predictors should be exactly orthogonal in this balanced 2x2x2 design
    corr = sb[["pos_dep_first", "role_dep_reactive", "decl_dep_first"]].corr()
    off_diag_max = (corr.values - np.eye(3)).__abs__().max()
    assert off_diag_max < 1e-9, f"{model_key}: predictors not orthogonal, max |corr|={off_diag_max:.2e}"

    stage_b_frames[model_key] = sb
    print(f"{model_key}: {len(sb)} rows, predictors orthogonal (max |corr|={off_diag_max:.2e})")


llama3.2-1b: 360 rows, predictors orthogonal (max |corr|=1.17e-17)
gemma3-1b-pt: 360 rows, predictors orthogonal (max |corr|=1.17e-17)


## Step 6 — Stage B 주효과 모델: Mixed model (main) + pair-clustered OLS (비교용)

```
D ~ pos_dep_first + role_dep_reactive + decl_dep_first
```

- **MixedLM** (`statsmodels.formula.api.mixedlm`, `groups=pair_id`, random intercept): pair 내
  8 cell의 비독립성을 모델 구조 자체로 반영한다. main으로 본다.
- **Pair-clustered OLS** (`statsmodels.api.OLS`, `cov_type="cluster"`): 같은 점추정을 (균형
  설계이므로 OLS 점추정은 MixedLM의 고정효과 추정과 거의 같다) pair 단위로 클러스터링한 표준오차로
  다시 본다. 비교용이다.

둘 중 어느 쪽을 메인으로 볼지는 연구자가 정한다 — 이 노트북은 MixedLM을 기본값으로 제시할 뿐
강제하지 않는다.

**확인할 것:** 두 모델 모두 `Converged: True`인지(아니면 출력에 표시되는 수렴 경고를 그대로
보고하라 — 이 노트북이 자동으로 재시도하지 않는다).


In [8]:
MAIN_EFFECTS_FORMULA_RHS = "pos_dep_first + role_dep_reactive + decl_dep_first"


def fit_mixedlm(sb, formula_rhs):
    md_ = smf.mixedlm(f"D ~ {formula_rhs}", sb, groups=sb["pair_id"])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        mdf = md_.fit()
    return mdf, caught


def fit_clustered_ols(sb, predictor_cols):
    X = sm.add_constant(sb[predictor_cols])
    return sm.OLS(sb["D"], X).fit(cov_type="cluster", cov_kwds={"groups": sb["pair_id"]})


def mixedlm_coef_table(mdf, model_key, spec_label):
    fe_idx = mdf.fe_params.index
    ci = mdf.conf_int().loc[fe_idx]
    return pd.DataFrame({
        "model": model_key,
        "model_spec": spec_label,
        "method": "mixedlm",
        "term": fe_idx,
        "estimate": mdf.fe_params.values,
        "std_err": mdf.bse_fe.values,
        "ci_low": ci[0].values,
        "ci_high": ci[1].values,
        "p_value": mdf.pvalues.loc[fe_idx].values,
    })


def ols_coef_table(ols_res, model_key, spec_label):
    ci = ols_res.conf_int()
    return pd.DataFrame({
        "model": model_key,
        "model_spec": spec_label,
        "method": "ols_cluster",
        "term": ols_res.params.index.map(lambda t: "Intercept" if t == "const" else t),
        "estimate": ols_res.params.values,
        "std_err": ols_res.bse.values,
        "ci_low": ci[0].values,
        "ci_high": ci[1].values,
        "p_value": ols_res.pvalues.values,
    })


main_predictor_cols = ["pos_dep_first", "role_dep_reactive", "decl_dep_first"]
main_tables = []
convergence_log = []

for model_key in MODEL_KEYS:
    sb = stage_b_frames[model_key]

    mdf, warns = fit_mixedlm(sb, MAIN_EFFECTS_FORMULA_RHS)
    convergence_log.append({
        "model": model_key, "model_spec": "main_effects",
        "converged": mdf.converged, "n_warnings": len(warns),
    })
    main_tables.append(mixedlm_coef_table(mdf, model_key, "main_effects"))

    ols_res = fit_clustered_ols(sb, main_predictor_cols)
    main_tables.append(ols_coef_table(ols_res, model_key, "main_effects"))

    print(f"{model_key}: MixedLM converged={mdf.converged} ({len(warns)} fit warning(s))")

convergence_df = pd.DataFrame(convergence_log)
convergence_df


llama3.2-1b: MixedLM converged=True (0 fit warning(s))
gemma3-1b-pt: MixedLM converged=True (0 fit warning(s))


,model,model_spec,converged,n_warnings
0,llama3.2-1b,main_effects,True,0
1,gemma3-1b-pt,main_effects,True,0


## Step 7 — Stage B 주효과 계수 표

두 방법(MixedLM, clustered OLS)의 추정치가 균형설계 하에서 서로 가깝게 나오는지 직접 비교한다.
크게 갈리면(특히 표준오차가 크게 다르면) pair-level dependence 처리 방식이 결과에 실질적으로
영향을 준다는 뜻이므로 그 자체를 보고해야 한다.


In [9]:
main_effects_df = pd.concat(main_tables, ignore_index=True)

pivot_estimate = main_effects_df.pivot_table(
    index=["model", "term"], columns="method", values="estimate"
)
pivot_se = main_effects_df.pivot_table(
    index=["model", "term"], columns="method", values="std_err"
)
print("=== estimate: mixedlm vs ols_cluster ===")
print(pivot_estimate.round(4))
print()
print("=== std_err: mixedlm vs ols_cluster ===")
print(pivot_se.round(4))

main_effects_df


=== estimate: mixedlm vs ols_cluster ===
method                          mixedlm  ols_cluster
model        term                                   
gemma3-1b-pt Intercept           0.5894       0.5894
             decl_dep_first      0.2002       0.2002
             pos_dep_first       1.1432       1.1432
             role_dep_reactive   0.3605       0.3605
llama3.2-1b  Intercept           0.4397       0.4397
             decl_dep_first      0.4114       0.4114
             pos_dep_first       1.2272       1.2272
             role_dep_reactive   1.0713       1.0713

=== std_err: mixedlm vs ols_cluster ===
method                          mixedlm  ols_cluster
model        term                                   
gemma3-1b-pt Intercept           0.0877       0.0881
             decl_dep_first      0.0265       0.0227
             pos_dep_first       0.0265       0.0354
             role_dep_reactive   0.0265       0.0457
llama3.2-1b  Intercept           0.0815       0.0819
             decl

,model,model_spec,method,term,estimate,std_err,ci_low,ci_high,p_value
0,llama3.2-1b,main_effects,mixedlm,Intercept,0.439694,0.081542,0.279874,0.599514,6.958898e-08
1,llama3.2-1b,main_effects,mixedlm,pos_dep_first,1.227176,0.024295,1.179559,1.274794,0.000000e+00
2,llama3.2-1b,main_effects,mixedlm,role_dep_reactive,1.071337,0.024295,1.023719,1.118954,0.000000e+00
3,llama3.2-1b,main_effects,mixedlm,decl_dep_first,0.411432,0.024295,0.363814,0.459050,2.494408e-64
4,llama3.2-1b,main_effects,ols_cluster,Intercept,0.439694,0.081886,0.279201,0.600187,7.891234e-08
5,llama3.2-1b,main_effects,ols_cluster,pos_dep_first,1.227176,0.037656,1.153372,1.300980,5.817505e-233
6,llama3.2-1b,main_effects,ols_cluster,role_dep_reactive,1.071337,0.036501,0.999796,1.142877,2.320616e-189
7,llama3.2-1b,main_effects,ols_cluster,decl_dep_first,0.411432,0.022916,0.366518,0.456346,4.447536e-72
8,gemma3-1b-pt,main_effects,mixedlm,Intercept,0.589417,0.087693,0.417542,0.761292,1.800088e-11
9,gemma3-1b-pt,main_effects,mixedlm,pos_dep_first,1.143175,0.026452,1.091330,1.195021,0.000000e+00


## Step 8 — 손-추정치와의 정합성 체크 (점검 신호용, claim 아님)

직전 단계에서 손으로 어림한 값과 비교한다: Llama P≈1.2 / R≈1.1, Gemma P≈1.1 / R≈0.44
(`P` = `pos_dep_first` 계수, `R` = `role_dep_reactive` 계수, MixedLM 추정치 기준).
**정합 여부 자체가 점검 신호다** — 크게 벗어나면 predictor 도출이나 데이터 어딘가에 문제가
있다는 뜻이고, 비슷하면 Step 5의 도출 로직이 맞게 작동했다는 교차 확인이 된다. 이 비교는
회귀의 정확도를 깎아내리거나 손-추정치를 검증하는 절차가 아니라, 둘 다 보여주는 것이다.


In [10]:
HAND_ESTIMATES = {
    "llama3.2-1b": {"pos_dep_first": 1.2, "role_dep_reactive": 1.1},
    "gemma3-1b-pt": {"pos_dep_first": 1.1, "role_dep_reactive": 0.44},
}

check_rows = []
mixedlm_main = main_effects_df[(main_effects_df["method"] == "mixedlm") & (main_effects_df["model_spec"] == "main_effects")]
for model_key, hand_vals in HAND_ESTIMATES.items():
    for term, hand_v in hand_vals.items():
        fit_v = mixedlm_main.loc[
            (mixedlm_main["model"] == model_key) & (mixedlm_main["term"] == term), "estimate"
        ].iloc[0]
        check_rows.append({
            "model": model_key,
            "term": term,
            "hand_estimate": hand_v,
            "mixedlm_estimate": fit_v,
            "abs_diff": abs(fit_v - hand_v),
        })

hand_check_df = pd.DataFrame(check_rows)
hand_check_df


,model,term,hand_estimate,mixedlm_estimate,abs_diff
0,llama3.2-1b,pos_dep_first,1.20,1.227176,0.027176
1,llama3.2-1b,role_dep_reactive,1.10,1.071337,0.028663
2,gemma3-1b-pt,pos_dep_first,1.10,1.143175,0.043175
3,gemma3-1b-pt,role_dep_reactive,0.44,0.360482,0.079518


## Step 9 — 확장 모델: position × useState-form 상호작용 (exploratory, 별도 제시)

주효과 모델은 이미 Step 6–8에서 보고했다. 여기서는 "position 효과 자체가 useState-form 여부에
따라 달라지는가"(또는 그 반대)를 보는 상호작용항 하나를 추가해 **별도로** 본다. 이 모델은 메인
결과를 대체하지 않고, 주효과 추정치가 상호작용을 넣어도 안정적인지 확인하는 추가 진단이다.


In [11]:
INTERACTION_FORMULA_RHS = f"{MAIN_EFFECTS_FORMULA_RHS} + pos_dep_first:role_dep_reactive"
interaction_predictor_cols = main_predictor_cols  # OLS interaction col added separately below

interaction_tables = []
for model_key in MODEL_KEYS:
    sb = stage_b_frames[model_key].copy()
    sb["pos_x_role"] = sb["pos_dep_first"] * sb["role_dep_reactive"]

    mdf_int, warns_int = fit_mixedlm(sb, INTERACTION_FORMULA_RHS)
    convergence_log.append({
        "model": model_key, "model_spec": "with_interaction",
        "converged": mdf_int.converged, "n_warnings": len(warns_int),
    })
    interaction_tables.append(mixedlm_coef_table(mdf_int, model_key, "with_interaction"))

    ols_int = fit_clustered_ols(sb, main_predictor_cols + ["pos_x_role"])
    interaction_tables.append(ols_coef_table(ols_int, model_key, "with_interaction"))

    print(f"{model_key} (+interaction): MixedLM converged={mdf_int.converged} ({len(warns_int)} fit warning(s))")

interaction_df = pd.concat(interaction_tables, ignore_index=True)
interaction_df


llama3.2-1b (+interaction): MixedLM converged=True (0 fit warning(s))
gemma3-1b-pt (+interaction): MixedLM converged=True (0 fit warning(s))


,model,model_spec,method,term,estimate,std_err,ci_low,ci_high,p_value
0,llama3.2-1b,with_interaction,mixedlm,Intercept,0.439694,0.081542,0.279874,0.599513,6.958753e-08
1,llama3.2-1b,with_interaction,mixedlm,pos_dep_first,1.227176,0.024253,1.179640,1.274712,0.000000e+00
2,llama3.2-1b,with_interaction,mixedlm,role_dep_reactive,1.071337,0.024253,1.023801,1.118873,0.000000e+00
3,llama3.2-1b,with_interaction,mixedlm,decl_dep_first,0.411432,0.024253,0.363896,0.458968,1.521141e-64
4,llama3.2-1b,with_interaction,mixedlm,pos_dep_first:role_dep_reactive,-0.034916,0.024253,-0.082452,0.012620,1.499753e-01
5,llama3.2-1b,with_interaction,ols_cluster,Intercept,0.439694,0.082001,0.278975,0.600413,8.228223e-08
6,llama3.2-1b,with_interaction,ols_cluster,pos_dep_first,1.227176,0.037709,1.153268,1.301084,2.589202e-232
7,llama3.2-1b,with_interaction,ols_cluster,role_dep_reactive,1.071337,0.036552,0.999696,1.142978,7.792652e-189
8,llama3.2-1b,with_interaction,ols_cluster,decl_dep_first,0.411432,0.022948,0.366455,0.456409,7.004080e-72
9,llama3.2-1b,with_interaction,ols_cluster,pos_x_role,-0.034916,0.018291,-0.070765,0.000934,5.627206e-02


## Step 10 — Stage B 표 저장

주효과 모델과 상호작용 모델 결과를 `model_spec` 컬럼으로 구분해 하나의 CSV에 합쳐 저장한다.


In [12]:
regression_df = pd.concat([main_effects_df, interaction_df], ignore_index=True)
regression_df.to_csv("reactivity_regression.csv", index=False)
print(f"wrote reactivity_regression.csv: {len(regression_df)} rows")
regression_df


wrote reactivity_regression.csv: 36 rows


,model,model_spec,method,term,estimate,std_err,ci_low,ci_high,p_value
0,llama3.2-1b,main_effects,mixedlm,Intercept,0.439694,0.081542,0.279874,0.599514,6.958898e-08
1,llama3.2-1b,main_effects,mixedlm,pos_dep_first,1.227176,0.024295,1.179559,1.274794,0.000000e+00
2,llama3.2-1b,main_effects,mixedlm,role_dep_reactive,1.071337,0.024295,1.023719,1.118954,0.000000e+00
3,llama3.2-1b,main_effects,mixedlm,decl_dep_first,0.411432,0.024295,0.363814,0.459050,2.494408e-64
4,llama3.2-1b,main_effects,ols_cluster,Intercept,0.439694,0.081886,0.279201,0.600187,7.891234e-08
5,llama3.2-1b,main_effects,ols_cluster,pos_dep_first,1.227176,0.037656,1.153372,1.300980,5.817505e-233
6,llama3.2-1b,main_effects,ols_cluster,role_dep_reactive,1.071337,0.036501,0.999796,1.142877,2.320616e-189
7,llama3.2-1b,main_effects,ols_cluster,decl_dep_first,0.411432,0.022916,0.366518,0.456346,4.447536e-72
8,gemma3-1b-pt,main_effects,mixedlm,Intercept,0.589417,0.087693,0.417542,0.761292,1.800088e-11
9,gemma3-1b-pt,main_effects,mixedlm,pos_dep_first,1.143175,0.026452,1.091330,1.195021,0.000000e+00


## Step 11 — `reactivity_inference_summary.md` 작성

Stage A·B 결과, 두 검정/두 모델/두 추정방법의 일치 여부, 손-추정치 교차 확인, 상호작용 모델,
limitation을 하나의 markdown으로 정리한다. 최종 판정 문장은 쓰지 않는다.


In [13]:
def fmt(x, nd=4):
    if pd.isna(x):
        return "NaN"
    return f"{x:.{nd}f}"


lines = []
lines.append("# Reactivity Pilot -- Statistical Inference Summary")
lines.append("")
lines.append(
    "범위: `reactivity_logits_3tokens.csv`(이미 측정된 logit)만 입력으로 쓰는 통계 분석이다. "
    "추가 logit 측정이나 activation patching은 포함하지 않는다."
)
lines.append("")
lines.append(
    "`reactive_var`/`stable_var`/`dep`/`alt`는 모두 useState/const 선언 형태에 붙인 라벨이며, "
    "reactivity 그 자체를 측정한 것이 아니다. 이 문서가 지지하는 가장 강한 주장은 "
    "\"useState-form / declared-reactivity sensitivity\"이며, \"React reactivity 의미 이해\" "
    "수준의 주장은 하지 않는다."
)
lines.append("")

lines.append("## 0. 데이터 및 균형설계 확인")
lines.append("")
lines.append(
    f"입력: `reactivity_logits_3tokens.csv`, {len(df)} row ({n_pairs} pair x 8 cell x "
    f"{len(MODEL_KEYS)} model). 모델별 모든 pair가 state_role/decl_order/body_order/"
    f"reactive_body_pos 네 축 모두 4/4로 균형 잡힌 완전요인설계임을 확인했다."
)
lines.append("")

lines.append("## 1. Stage A -- pair-level 검정 (position 등 상쇄 후 useState-form 효과)")
lines.append("")
lines.append(
    "각 (model, pair)의 8 cell `LD_reactive` 평균(pair-mean)이 0보다 큰지 단측 검정했다. "
    "이 평균은 state_role/decl_order/body_order 4/4 균형 덕분에 position·decl_order·"
    "identity bias가 상쇄된 값이다."
)
lines.append("")
lines.append(
    "| model | n pairs | mean | median | sd | frac>0 | t (one-sided p) | Cohen's d | "
    "mean 95% CI | Wilcoxon (one-sided p) | rank-biserial r | median bootstrap 95% CI |"
)
lines.append("|---|---|---|---|---|---|---|---|---|---|---|---|")
for _, r in stage_a_df.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {int(r['n_pairs'])} | {fmt(r['pair_mean_mean'])} | "
        f"{fmt(r['pair_mean_median'])} | {fmt(r['pair_mean_sd'])} | {r['frac_pair_mean_pos']:.3f} | "
        f"{fmt(r['t_stat'])} ({r['t_p_onesided_greater']:.3e}) | {fmt(r['cohend'])} | "
        f"[{fmt(r['mean_ci95_low'])}, {fmt(r['mean_ci95_high'])}] | "
        f"{fmt(r['wilcoxon_stat'])} ({r['wilcoxon_p_onesided_greater']:.3e}) | "
        f"{fmt(r['rank_biserial_r'])} | "
        f"[{fmt(r['median_boot_ci95_low'])}, {fmt(r['median_boot_ci95_high'])}] |"
    )
lines.append("")
lines.append(
    f"양측 p값(참고용): " + "; ".join(
        f"{MODEL_LABELS[r['model']]} t={r['t_p_twosided']:.3e}, "
        f"Wilcoxon={r['wilcoxon_p_twosided']:.3e}"
        for _, r in stage_a_df.iterrows()
    )
)
lines.append("")
lines.append(
    f"다중비교: 모델 {len(MODEL_KEYS)}개 x 검정 2개(t-test, Wilcoxon) = {N_COMPARISONS}개의 단측 "
    f"검정을 했다. 보정하지 않은 raw p값을 그대로 보고했다. 참고용 Bonferroni 임계값(alpha=0.05): "
    f"{bonferroni_alpha:.5f}."
)
lines.append("")
lines.append("### 두 검정/두 모델 일치 여부")
lines.append("")
lines.append("| model | t-test p (one-sided) | Wilcoxon p (one-sided) | 둘 다 p<0.05 | 둘 다 p<Bonferroni |")
lines.append("|---|---|---|---|---|")
for _, r in agreement_df.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {r['t_p_onesided']:.3e} | {r['wilcoxon_p_onesided']:.3e} | "
        f"{r['both_sig_at_0.05']} | {r['both_sig_at_bonferroni']} |"
    )
lines.append("")

lines.append("## 2. Stage B -- 회귀 분해 (주효과 모델)")
lines.append("")
lines.append(
    "`D = logit(dep) - logit(alt)`를 `pos_dep_first`(position) + `role_dep_reactive`"
    "(useState-form) + `decl_dep_first`(declaration order)로 회귀했다. 세 predictor 모두 "
    "+-1 코딩이며, 균형 2x2x2 설계에서 서로 완전히 직교함을 확인했다(상호 상관 ~0)."
)
lines.append("")
lines.append("| model | method | term | estimate | std err | 95% CI | p value |")
lines.append("|---|---|---|---|---|---|---|")
for _, r in main_effects_df.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {r['method']} | {r['term']} | {fmt(r['estimate'])} | "
        f"{fmt(r['std_err'])} | [{fmt(r['ci_low'])}, {fmt(r['ci_high'])}] | {r['p_value']:.3e} |"
    )
lines.append("")
lines.append(
    "MixedLM 수렴 여부:\n" + "\n".join(
        f"- {MODEL_LABELS[r['model']]} ({r['model_spec']}): converged={r['converged']}, "
        f"fit warning {int(r['n_warnings'])}건"
        for _, r in convergence_df.iterrows()
    )
)
lines.append("")
lines.append(
    "**해석 메모(자동 판정 아님):** `role_dep_reactive` 계수가 useState-form / declared-reactivity "
    "효과의 크기다. `pos_dep_first`는 같은 단위의 순수 position(surface order) 효과이며, "
    "`role_dep_reactive`와 비교 가능하다 -- 어느 쪽이 더 크고 작은지는 위 표의 estimate와 CI를 "
    "보고 연구자가 판단한다. `decl_dep_first`는 선언 순서라는 2차 nuisance factor이고, 절편은 "
    "position/role/decl_order로 설명되지 않는 dep/alt identity 자체의 잔차 편향이다."
)
lines.append("")

lines.append("## 3. 손-추정치 교차 확인 (점검 신호, claim 아님)")
lines.append("")
lines.append(
    "Stage B 회귀계수가 사전에 손으로 어림한 값과 대체로 정합하는지 보는 점검 셀이다. 정합하지 "
    "않는 항목이 있다면 그 자체가 데이터나 도출 로직을 다시 볼 신호로 남겨둔다."
)
lines.append("")
lines.append("| model | term | hand estimate | MixedLM estimate | abs diff |")
lines.append("|---|---|---|---|---|")
for _, r in hand_check_df.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {r['term']} | {fmt(r['hand_estimate'], 2)} | "
        f"{fmt(r['mixedlm_estimate'])} | {fmt(r['abs_diff'])} |"
    )
lines.append("")

lines.append("## 4. 확장 모델 -- position x useState-form 상호작용 (exploratory)")
lines.append("")
lines.append(
    "주효과 모델에 `pos_dep_first:role_dep_reactive` 상호작용항을 추가한 결과다. 메인 결과를 "
    "대체하지 않는 추가 진단이다."
)
lines.append("")
lines.append("| model | method | term | estimate | std err | 95% CI | p value |")
lines.append("|---|---|---|---|---|---|---|")
for _, r in interaction_df.iterrows():
    lines.append(
        f"| {MODEL_LABELS[r['model']]} | {r['method']} | {r['term']} | {fmt(r['estimate'])} | "
        f"{fmt(r['std_err'])} | [{fmt(r['ci_low'])}, {fmt(r['ci_high'])}] | {r['p_value']:.3e} |"
    )
lines.append("")

lines.append("## 5. Limitation")
lines.append("")
lines.append(
    "1. `reactive_var`/`stable_var`/`dep`/`alt`는 useState/const 선언 형태에 붙인 라벨이며 "
    "reactivity 그 자체를 측정한 것이 아니다. 유의한 양의 `role_dep_reactive` 효과가 나와도 "
    "허용 claim은 \"useState-form / declared-reactivity sensitivity\"까지다."
)
lines.append(
    "2. useState 배정에는 setter(`setX`)와 destructuring(`const [X, setX] = ...`) 구조가 항상 "
    "동반된다. 이 design은 reactivity 자체와 setter/destructuring 형태를 분리하지 못한다."
)
lines.append(
    "3. Stage A/B 모두 `useEffect` 단일 condition, dep/alt 변수 pair 풀, prefix-only 입력이라는 "
    "동일한 데이터셋 범위 안에서만 추정한 것이다."
)
lines.append("4. 1B 모델(Llama 3.2 1B, Gemma 3 1B-pt) 두 개에 대해서만 추정했다.")
lines.append(
    "5. Stage A의 검정과 Stage B의 회귀는 같은 데이터를 다른 방식으로 본 것이라 서로 독립적인 "
    "증거가 아니다 -- 둘이 같은 방향을 보여주는 것은 일관성 확인이지 독립 재현이 아니다."
)
lines.append(
    "6. 다중비교(Stage A의 2 모델 x 2 검정)를 보정하지 않았다 -- raw p값을 그대로 제시했다."
)
lines.append("")

lines.append("## 6. 출력 파일")
lines.append("")
lines.append(f"- `reactivity_pairlevel_test.csv`: Stage A 결과, {len(stage_a_df)} row.")
lines.append(f"- `reactivity_regression.csv`: Stage B 계수(주효과 + 상호작용, MixedLM + OLS), {len(regression_df)} row.")
lines.append("- `reactivity_inference_summary.md`: 본 문서.")
lines.append("")

with open("reactivity_inference_summary.md", "w") as f:
    f.write("\n".join(lines))

print("wrote reactivity_inference_summary.md")


wrote reactivity_inference_summary.md


## 결론 (작성 시점 -- 미실행)

이 노트북 파일 자체는 실행되지 않은 채로 전달되었다(핵심 로직은 별도 환경에서 실제 CSV로
사전 검증됨 -- 상단 참고). 위에서부터 순서대로 셀을 돌리면서 Step 1의 균형설계 assert, Step 6/9의
MixedLM 수렴 여부, Step 8의 손-추정치 정합성부터 확인하라.

- Stage A(검정)와 Stage B(회귀)는 같은 결론을 다른 각도에서 보여줘야 한다 -- 둘이 어긋나면
  (예: Stage A는 유의하지만 Stage B의 `role_dep_reactive`가 `pos_dep_first`보다 작다면) 그 자체가
  보고할 결과다.
- 최종 메인 claim의 표현은 연구자가 결정한다. 이 노트북은 판정하지 않는다.
- 출력 파일: `reactivity_pairlevel_test.csv`, `reactivity_regression.csv`,
  `reactivity_inference_summary.md`.
